# AirGraph Joint Training

Joint training of AeroGPT + GATv2 graph attention layer.

**Prerequisites**: Pre-trained AeroGPT checkpoint from notebook 01.
**Estimated time**: 4 hours for 30K steps.

In [ ]:
# Cell 1: Setup (same as notebook 01)
import subprocess, sys, os
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torch-geometric', 'polars', 'pyarrow', 'pyopensky',
    'structlog', 'pyyaml', 'scipy', 'matplotlib', 'tqdm', 'rich', 'httpx'])

if os.path.exists('/content'):
    if not os.path.exists('aeroconform'):
        subprocess.check_call(['git', 'clone', 'https://github.com/Nadosaurusrex/aeroconform.git'])
    os.chdir('aeroconform')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
    sys.path.insert(0, '.')

from pathlib import Path
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/aeroconform/data')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/aeroconform/checkpoints')
else:
    DATA_DIR = Path('data')
    CHECKPOINT_DIR = Path('checkpoints')

print('Setup complete')

In [ ]:
# Cell 2: Load data and pre-trained model
import torch
import polars as pl
from torch.utils.data import DataLoader, random_split
from src.models.aerogpt import AeroGPT
from src.models.airgraph import AirGraph
from src.models.combined import AeroConformModel
from src.data.dataset import TrajectoryMapDataset, collate_trajectories
from src.data.flight_segmentation import segment_flights
from src.data.preprocessing import delta_encode, compute_norm_stats
from src.data.schemas import NormStats
from src.utils.config import ModelConfig, TrainingConfig, GraphConfig
from src.utils.logging import setup_logging

setup_logging(level='INFO')

# Load configs
model_config = ModelConfig.from_yaml(Path('configs/model.yaml'))
graph_config = GraphConfig.from_yaml(Path('configs/graph.yaml'))

# Load pre-trained AeroGPT
combined = AeroConformModel(model_config, graph_config)
ckpt = torch.load(CHECKPOINT_DIR / 'aerogpt_final.pt', map_location='cpu', weights_only=False)
combined.aerogpt.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded pre-trained AeroGPT')

# Load data
norm_stats = NormStats.load(str(DATA_DIR / 'norm_stats.npz'))
parquet_files = sorted((DATA_DIR / 'raw').glob('*.parquet'))
all_flights = []
for pf in parquet_files:
    df = pl.read_parquet(pf)
    flights = segment_flights(df)
    all_flights.extend(flights)

dataset = TrajectoryMapDataset(all_flights, context_length=128, norm_stats=norm_stats)
val_size = max(1, len(dataset) // 10)
train_ds, val_ds = random_split(dataset, [len(dataset) - val_size, val_size])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_trajectories)
val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate_trajectories)
print(f'Dataset: {len(dataset)} windows, {len(train_loader)} train batches')

In [ ]:
# Cell 3: Joint training loop (30K steps)
import copy
from src.models.losses import gaussian_nll_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
combined = combined.to(device)

optimizer = torch.optim.AdamW(combined.parameters(), lr=1e-4, weight_decay=0.01)
max_steps = 30000
step = 0
losses = []

data_iter = iter(train_loader)
combined.train()

while step < max_steps:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)
    
    x = batch['input'].to(device)
    target = batch['target'].to(device)
    time_gaps = batch['time_gaps'].to(device)
    mask = batch['mask'].to(device)
    
    optimizer.zero_grad()
    
    # Foundation model forward
    means, log_vars, hidden = combined.forward_foundation(x, time_gaps, mask)
    foundation_loss = gaussian_nll_loss(means, log_vars, target, mask)
    
    loss = foundation_loss  # Graph loss added when graph data available
    loss.backward()
    torch.nn.utils.clip_grad_norm_(combined.parameters(), 1.0)
    optimizer.step()
    
    losses.append(loss.item())
    step += 1
    
    if step % 500 == 0:
        print(f'Step {step}/{max_steps}, Loss: {loss.item():.4f}')
    
    if step % 5000 == 0:
        torch.save({
            'model_state_dict': combined.state_dict(),
            'step': step,
            'model_config': model_config,
            'graph_config': graph_config,
        }, CHECKPOINT_DIR / f'combined_step_{step}.pt')

# Save final
torch.save({
    'model_state_dict': combined.state_dict(),
    'model_config': model_config,
    'graph_config': graph_config,
}, CHECKPOINT_DIR / 'combined_final.pt')
print('Joint training complete')

In [ ]:
# Cell 4: Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.title('Joint Training Loss')
plt.xlabel('Step')
plt.ylabel('NLL')
plt.yscale('log')
plt.savefig(str(CHECKPOINT_DIR / 'joint_training_curve.png'), dpi=150)
plt.show()